# Expand Trigger Strategy Sweep on GSM8K (v2 — velocity / block_length)

Velocity 分母改为 `block_length`（线性化），仅跑 velocity sweep + 3 个 baseline，共 12 个任务：

- **Baseline — dualcache only (1)**：ratio 模式 mtr=0.0（纯 dual_cache，永不扩展）
- **Baseline — ratio 0.5 (1)**：ratio 模式 mtr=0.5
- **Baseline — ratio 0.9 (1)**：ratio 模式 mtr=0.9
- **Velocity (9)**：`transferred / block_length >= threshold`（线性解码速率）

固定参数：gen_length=128, steps=128, block_length=32, threshold=0.9,
dual_cache=True, mid_block_expand=True, rewarm_on_expand=True, seed=42, 全量 GSM8K。
GPU 池：6 卡（0-5），任务队列自动调度。

## 1. 环境设置

In [ ]:
import os, torch, gc

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} '
          f'({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)')

## 2. 任务配置（3 baselines + 9 velocity = 12 任务）

In [ ]:
import subprocess, datetime

task = 'gsm8k'
fewshot = 5
seed = 42
gen_length = 128
steps = 128
block_length = 32
threshold = 0.9
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

SWEEP_RANGE = [round(x * 0.1, 1) for x in range(1, 10)]  # 0.1 ~ 0.9

TASK_CONFIGS = [
    # --- Baselines ---
    {'name': 'baseline_dualcache', 'expand_trigger_mode': 'ratio', 'mid_trigger_ratio': 0.0,
     'vel_threshold': 0.5, 'conf_threshold': 0.5},
    {'name': 'baseline_ratio05', 'expand_trigger_mode': 'ratio', 'mid_trigger_ratio': 0.5,
     'vel_threshold': 0.5, 'conf_threshold': 0.5},
    {'name': 'baseline_ratio09', 'expand_trigger_mode': 'ratio', 'mid_trigger_ratio': 0.9,
     'vel_threshold': 0.5, 'conf_threshold': 0.5},
]
# --- Velocity sweep (denominator = block_length) ---
for v in SWEEP_RANGE:
    TASK_CONFIGS.append({
        'name': f'vel_{v:.1f}'.replace('.', ''),
        'expand_trigger_mode': 'velocity',
        'mid_trigger_ratio': 0.0, 'vel_threshold': v, 'conf_threshold': 0.5,
    })

GPU_POOL = [int(x) for x in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]

print(f'Total tasks: {len(TASK_CONFIGS)}  |  Available GPUs: {len(GPU_POOL)}')
print(f'Timestamp: {timestamp}')
print()
print(f'{"Name":<22} {"Mode":<16} {"MTR":<6} {"VelT":<6} {"ConfT":<6}')
print('-' * 56)
for cfg in TASK_CONFIGS:
    print(f'{cfg["name"]:<22} {cfg["expand_trigger_mode"]:<16} {cfg["mid_trigger_ratio"]:<6} {cfg["vel_threshold"]:<6} {cfg["conf_threshold"]:<6}')

## 3. 并行启动任务

In [ ]:
import threading, queue

task_queue = queue.Queue()
for cfg in TASK_CONFIGS:
    task_queue.put(cfg)

results_lock = threading.Lock()
all_results = []


def gpu_worker(gpu_id):
    while True:
        try:
            cfg = task_queue.get_nowait()
        except queue.Empty:
            return

        name = cfg['name']
        log_file = f'nlogs/sweep_exptrig_{task}_{name}_{timestamp}.log'
        output_dir = f'evals_results/expand_trigger/{task}-{name}-{timestamp}'
        records_dir = f'{output_dir}/step_records'

        model_args = ','.join([
            "model_path='GSAI-ML/LLaDA-8B-Instruct'",
            f'gen_length={gen_length}',
            f'steps={steps}',
            f'block_length={block_length}',
            f'threshold={threshold}',
            'use_cache=True',
            'show_speed=True',
            f"step_records_dir='{records_dir}'",
            f'seed={seed}',
            'dual_cache=True',
            'mid_block_expand=True',
            f"mid_trigger_ratio={cfg['mid_trigger_ratio']}",
            'rewarm_on_expand=True',
            'front_block_fallback_only=True',
            f"expand_trigger_mode={cfg['expand_trigger_mode']}",
            f"vel_threshold={cfg['vel_threshold']}",
            f"conf_threshold={cfg['conf_threshold']}",
        ])

        cmd = (
            f'CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py '
            f'--tasks {task} --num_fewshot {fewshot} '
            f'--confirm_run_unsafe_code --model llada_dist '
            f'--model_args {model_args} '
            f'--output_path {output_dir} --log_samples'
        )

        print(f'[GPU {gpu_id}] START  {name}')

        p = subprocess.Popen(
            cmd, shell=True,
            stdout=open(log_file, 'w'),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()

        status = 'OK' if rc == 0 else f'FAILED(exit={rc})'
        print(f'[GPU {gpu_id}] DONE   {name}  {status}')

        with results_lock:
            all_results.append((cfg, name, log_file, output_dir, rc))

        task_queue.task_done()


threads = []
for gpu_id in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gpu_id,), daemon=True)
    t.start()
    threads.append(t)

print(f'\nLaunched {len(threads)} GPU workers for {len(TASK_CONFIGS)} tasks.')
print('Waiting for all tasks to complete...')

In [ ]:
for t in threads:
    t.join()

print(f'\nAll {len(all_results)} / {len(TASK_CONFIGS)} tasks finished.')
for cfg, name, log_file, output_dir, rc in all_results:
    status = 'OK' if rc == 0 else f'FAILED(exit={rc})'
    print(f'  {name:22s}  {status}  log={log_file}')

In [ ]:
# --- 单独补跑 baseline_dualcache (mtr=0.0, 纯 dual_cache 不扩展) ---
import subprocess

_cfg = {'name': 'baseline_dualcache', 'expand_trigger_mode': 'ratio',
        'mid_trigger_ratio': 0.0, 'vel_threshold': 0.5, 'conf_threshold': 0.5}
_gpu = 0

_log = f'nlogs/sweep_exptrig_{task}_{_cfg["name"]}_{timestamp}.log'
_out = f'evals_results/expand_trigger/{task}-{_cfg["name"]}-{timestamp}'
_rec = f'{_out}/step_records'

_model_args = ','.join([
    "model_path='GSAI-ML/LLaDA-8B-Instruct'",
    f'gen_length={gen_length}', f'steps={steps}',
    f'block_length={block_length}', f'threshold={threshold}',
    'use_cache=True', 'show_speed=True',
    f"step_records_dir='{_rec}'", f'seed={seed}',
    'dual_cache=True', 'mid_block_expand=True',
    f"mid_trigger_ratio={_cfg['mid_trigger_ratio']}",
    'rewarm_on_expand=True', 'front_block_fallback_only=True',
    f"expand_trigger_mode={_cfg['expand_trigger_mode']}",
    f"vel_threshold={_cfg['vel_threshold']}",
    f"conf_threshold={_cfg['conf_threshold']}",
])

_cmd = (
    f'CUDA_VISIBLE_DEVICES={_gpu} accelerate launch eval_llada.py '
    f'--tasks {task} --num_fewshot {fewshot} '
    f'--confirm_run_unsafe_code --model llada_dist '
    f'--model_args {_model_args} '
    f'--output_path {_out} --log_samples'
)

print(f'Running: {_cfg["name"]} on GPU {_gpu}')
print(f'Log: {_log}')
p = subprocess.Popen(_cmd, shell=True, stdout=open(_log, 'w'), stderr=subprocess.STDOUT)
rc = p.wait()
print(f'Done: {_cfg["name"]}  exit={rc}')

## 4. 解析评测结果

In [ ]:
import re, json, glob
import pandas as pd

def find_log(cfg_name, ts=None):
    """优先用当前 timestamp，找不到则回退到最新的同名 log。"""
    if ts:
        exact = f'nlogs/sweep_exptrig_{task}_{cfg_name}_{ts}.log'
        if os.path.exists(exact):
            return exact
    candidates = sorted(
        glob.glob(f'nlogs/sweep_exptrig_{task}_{cfg_name}_*.log'),
        key=os.path.getmtime, reverse=True)
    return candidates[0] if candidates else None

parsed_results = []
for cfg in TASK_CONFIGS:
    name = cfg['name']
    log_file = find_log(name, timestamp)
    if log_file is None:
        print(f'WARNING: no log for {name}')
        continue
    with open(log_file, 'r') as f:
        content = f.read()

    flex_m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', content)
    strict_m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', content)
    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', content)
    nfe_m = re.search(r'Total NFE is (\d+)', content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', content)

    parsed_results.append({
        'name': name,
        'mode': cfg['expand_trigger_mode'],
        'vel_threshold': cfg['vel_threshold'],
        'conf_threshold': cfg['conf_threshold'],
        'flex_acc': float(flex_m.group(1)) if flex_m else None,
        'strict_acc': float(strict_m.group(1)) if strict_m else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    })

df = pd.DataFrame(parsed_results)

print(f'Velocity Sweep — GSM8K (timestamp={timestamp})')
print('=' * 100)
print(f'{"Name":<22} {"Mode":<14} {"Vel":<5} {"Conf":<6} {"FlexAcc":<10} {"StrictAcc":<11} {"Tok/s":<10} {"NFE":<10} {"Time(s)":<10}')
print('-' * 100)
for r in parsed_results:
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(r['total_nfe']) if r['total_nfe'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['name']:<22} {r['mode']:<14} {r['vel_threshold']:<5} {r['conf_threshold']:<6} {fa:<10} {sa:<11} {sp:<10} {nf:<10} {tm:<10}")

display(df)

## 5. 对比图：Accuracy / NFE / Speed

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_valid = df.dropna(subset=['flex_acc']).copy()
if df_valid.empty:
    print('No valid results to plot.')
else:
    SERIES = [
        ('velocity', 'vel_threshold', '#2196F3', 'o-', 'velocity'),
    ]

    BASELINES = [
        ('baseline_dualcache', '#607D8B', '--',  'dualcache only (mtr=0.0)'),
        ('baseline_ratio05',   '#FF9800', '--',  'ratio mtr=0.5'),
        ('baseline_ratio09',   '#F44336', '--',  'ratio mtr=0.9'),
    ]

    METRICS = [
        ('flex_acc', 'Accuracy'),
        ('total_nfe', 'Total NFE'),
        ('tok_per_sec', 'Tokens / sec'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax_idx, (col, ylabel) in enumerate(METRICS):
        ax = axes[ax_idx]

        for bl_name, bl_color, bl_ls, bl_label in BASELINES:
            bl_row = df_valid[df_valid['name'] == bl_name]
            if not bl_row.empty:
                bl_val = bl_row.iloc[0][col]
                if pd.notna(bl_val):
                    suffix = f' ({bl_val:.4f})' if col == 'flex_acc' else ''
                    ax.axhline(y=bl_val, color=bl_color, linestyle=bl_ls,
                               alpha=0.7, label=f'{bl_label}{suffix}')

        for mode, xcol, color, marker, label in SERIES:
            sub = df_valid[df_valid['mode'] == mode].sort_values(xcol)
            if sub.empty:
                continue
            xs = sub[xcol].values
            ys = sub[col].values
            ax.plot(xs, ys, marker, color=color, lw=2, ms=5, label=label, alpha=0.85)

        ax.set_xlabel('Threshold')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel} vs Threshold', fontweight='bold')
        ax.set_xticks([round(x * 0.1, 1) for x in range(1, 10)])
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)

    fig.suptitle(f'Velocity Sweep (GSM8K, vel = transferred/block_length, seed={seed})',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    os.makedirs('../eval_results', exist_ok=True)
    plt.savefig('../eval_results/expand_trigger_sweep_v2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: eval_results/expand_trigger_sweep_v2.png')

## 6. Step Records 分析（可选）

In [ ]:
from collections import defaultdict

def find_step_records(cfg_name, ts=None):
    """优先用当前 timestamp，找不到则回退到最新的同名目录。"""
    if ts:
        exact = f'evals_results/expand_trigger/{task}-{cfg_name}-{ts}/step_records/step_records.json'
        if os.path.exists(exact):
            return exact
    candidates = sorted(
        glob.glob(f'evals_results/expand_trigger/{task}-{cfg_name}-*/step_records/step_records.json'),
        key=os.path.getmtime, reverse=True)
    return candidates[0] if candidates else None

step_data = {}
for cfg in TASK_CONFIGS:
    name = cfg['name']
    rpath = find_step_records(name, timestamp)
    if rpath and os.path.exists(rpath):
        with open(rpath, 'r') as f:
            step_data[name] = json.load(f)
        print(f'  [OK]   {name}: {len(step_data[name])} samples')
    else:
        print(f'  [MISS] {name}')

print(f'\nLoaded step records for {len(step_data)} / {len(TASK_CONFIGS)} configs.')

In [ ]:
if step_data:
    n = len(step_data)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 5), sharey=True)
    if n == 1:
        axes = [axes]

    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, n))

    for ax_idx, (cname, samples) in enumerate(step_data.items()):
        ax = axes[ax_idx]
        step_transferred = defaultdict(list)
        for sample_records in samples:
            for rec in sample_records:
                step_transferred[rec['global_step']].append(rec['transferred'])

        max_step = max(step_transferred.keys()) if step_transferred else 0
        steps_range = list(range(max_step + 1))
        means = [np.mean(step_transferred[s]) if s in step_transferred else 0 for s in steps_range]

        ax.bar(steps_range, means, color=cmap[ax_idx], alpha=0.7)
        ax.set_xlabel('Global Step')

        res = next((r for r in parsed_results if r['name'] == cname), None)
        acc_s = f"Acc={res['flex_acc']:.4f}" if res and res['flex_acc'] else ''
        nfe_s = f"NFE={res['total_nfe']}" if res and res['total_nfe'] else ''
        ax.set_title(f'{cname}\n{acc_s}  {nfe_s}', fontsize=9)
        ax.set_xlim(-0.5, max_step + 0.5)

    axes[0].set_ylabel('Tokens Transferred')
    fig.suptitle('Per-Step Decoded Tokens by Trigger Strategy', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../eval_results/expand_trigger_step_dist.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: eval_results/expand_trigger_step_dist.png')
else:
    print('No step records available.')

## 7. 从已有结果重新加载（可选）

如果 notebook 内核重启，运行此 cell 从磁盘加载已有结果，然后重新运行 Section 5~6。
每个 config 独立查找最新的 log 文件，支持 baseline_dualcache 单独补跑的场景。

In [ ]:
import glob, re, json, os
import pandas as pd

task = 'gsm8k'
seed = 42

SWEEP_RANGE = [round(x * 0.1, 1) for x in range(1, 10)]

TASK_CONFIGS = [
    {'name': 'baseline_dualcache', 'expand_trigger_mode': 'ratio', 'mid_trigger_ratio': 0.0,
     'vel_threshold': 0.5, 'conf_threshold': 0.5},
    {'name': 'baseline_ratio05', 'expand_trigger_mode': 'ratio', 'mid_trigger_ratio': 0.5,
     'vel_threshold': 0.5, 'conf_threshold': 0.5},
    {'name': 'baseline_ratio09', 'expand_trigger_mode': 'ratio', 'mid_trigger_ratio': 0.9,
     'vel_threshold': 0.5, 'conf_threshold': 0.5},
]
for v in SWEEP_RANGE:
    TASK_CONFIGS.append({
        'name': f'vel_{v:.1f}'.replace('.', ''),
        'expand_trigger_mode': 'velocity',
        'mid_trigger_ratio': 0.0, 'vel_threshold': v, 'conf_threshold': 0.5,
    })

def find_latest_log(task_name, log_dir='nlogs'):
    """为每个 config 独立找最新的 log 文件（支持不同 timestamp）。"""
    pattern = f'{log_dir}/sweep_exptrig_{task}_{task_name}_*.log'
    candidates = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
    return candidates[0] if candidates else None

parsed_results = []
timestamps_used = set()
for cfg in TASK_CONFIGS:
    name = cfg['name']
    log_file = find_latest_log(name)
    if log_file is None:
        print(f'  [MISS] {name}')
        continue

    ts = os.path.basename(log_file).replace(f'sweep_exptrig_{task}_{name}_', '').replace('.log', '')
    timestamps_used.add(ts)

    with open(log_file, 'r') as f:
        content = f.read()

    flex_m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', content)
    strict_m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', content)
    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', content)
    nfe_m = re.search(r'Total NFE is (\d+)', content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', content)

    parsed_results.append({
        'name': name,
        'mode': cfg['expand_trigger_mode'],
        'vel_threshold': cfg['vel_threshold'],
        'conf_threshold': cfg['conf_threshold'],
        'timestamp': ts,
        'flex_acc': float(flex_m.group(1)) if flex_m else None,
        'strict_acc': float(strict_m.group(1)) if strict_m else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    })
    print(f'  [OK]   {name:<22s}  ts={ts}')

df = pd.DataFrame(parsed_results)
print(f'\nLoaded {len(parsed_results)} results  (seed={seed}, timestamps={sorted(timestamps_used)})')
print(f'\n{"Name":<22} {"Mode":<14} {"FlexAcc":<10} {"NFE":<10} {"Tok/s":<10} {"Time(s)":<10}')
print('-' * 76)
for r in parsed_results:
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    nf = str(r['total_nfe']) if r['total_nfe'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['name']:<22} {r['mode']:<14} {fa:<10} {nf:<10} {sp:<10} {tm:<10}")

print(f'\nRe-run Section 5~6 cells to regenerate plots.')